# 14 - Gold: Dashboard Core Marts

This notebook creates the first dashboard-ready gold tables.

Outputs:

- `gold.country_year_observatory`
- `gold.country_latest_snapshot`
- `gold.bloc_year_observatory`
- `gold.bloc_latest_snapshot`

Gold tables are deliberately wide and dashboard-friendly. They combine macro, trade, ACLED conflict, FSI fragility, and coverage flags so Streamlit can render panels without repeatedly joining silver tables.

In [ ]:
# Cell 1 - Configuration
from datetime import datetime, timezone

from pyspark.sql import Window
from pyspark.sql import functions as F

spark.sql("USE CATALOG cemac_ecowas_aes_trade")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

START_YEAR = 1990
END_YEAR = 2024
loaded_at = datetime.now(timezone.utc)

print("Target gold tables:")
print("  gold.country_year_observatory")
print("  gold.country_latest_snapshot")
print("  gold.bloc_year_observatory")
print("  gold.bloc_latest_snapshot")

In [ ]:
# Cell 2 - Input table checks
def table_exists(schema_name, table_name):
    return spark.sql(f"SHOW TABLES IN {schema_name} LIKE '{table_name}'").count() > 0


required_tables = [
    ("audit", "country_year_source_coverage"),
    ("silver", "fact_macro_annual"),
    ("silver", "fact_trade_partner_annual"),
    ("silver", "trade_partner_concentration_annual"),
    ("silver", "fact_acled_country_year"),
    ("silver", "fact_fsi_annual"),
]

optional_tables = [
    ("silver", "comtrade_country_year_coverage"),
    ("silver", "comtrade_hs2_annual_w00"),
    ("silver", "comtrade_product_coverage"),
    ("audit", "comtrade_coverage_summary"),
]

missing_tables = [f"{schema}.{table}" for schema, table in required_tables if not table_exists(schema, table)]
if missing_tables:
    raise ValueError(f"Missing required input table(s): {missing_tables}")

print("All required tables found.")
for schema_name, table_name in required_tables:
    print(f"  {schema_name}.{table_name}: {spark.table(f'{schema_name}.{table_name}').count():,} rows")

print("\nOptional Comtrade tables:")
for schema_name, table_name in optional_tables:
    if table_exists(schema_name, table_name):
        print(f"  ✓ {schema_name}.{table_name}: {spark.table(f'{schema_name}.{table_name}').count():,} rows")
    else:
        print(f"  ✗ {schema_name}.{table_name}: not present (Comtrade product charts will be unavailable)")

# Quick trade source breakdown
print("\nTrade source breakdown in silver.fact_trade_partner_annual:")
spark.sql("""
    SELECT trade_source, selected_source, COUNT(DISTINCT reporter_iso3) AS reporters, COUNT(*) AS rows
    FROM silver.fact_trade_partner_annual
    GROUP BY trade_source, selected_source
    ORDER BY trade_source
""").show(truncate=False)


In [ ]:
# Cell 3 - Trade helper tables: annual totals and top partner
trade_fact = spark.table("silver.fact_trade_partner_annual")

trade_year_df = (
    trade_fact
    .where(~F.col("is_world_aggregate"))
    .groupBy(F.col("reporter_iso3").alias("country_iso3"), "year")
    .agg(
        F.countDistinct("counterpart_iso3").alias("trade_partner_count"),
        F.sum(F.coalesce(F.col("exports_fob_usd"), F.lit(0.0))).alias("exports_fob_usd"),
        F.sum(F.coalesce(F.col("imports_cif_usd"), F.lit(0.0))).alias("imports_cif_usd"),
        F.sum(F.coalesce(F.col("total_trade_usd"), F.lit(0.0))).alias("total_trade_usd"),
        F.sum(F.coalesce(F.col("trade_balance_usd"), F.lit(0.0))).alias("trade_balance_usd"),
        F.sum(F.when(F.col("counterpart_is_project_country"), F.coalesce(F.col("total_trade_usd"), F.lit(0.0))).otherwise(F.lit(0.0))).alias("intra_project_trade_usd"),
    )
    .withColumn("exports_billions_usd", F.col("exports_fob_usd") / F.lit(1_000_000_000.0))
    .withColumn("imports_billions_usd", F.col("imports_cif_usd") / F.lit(1_000_000_000.0))
    .withColumn("total_trade_billions_usd", F.col("total_trade_usd") / F.lit(1_000_000_000.0))
    .withColumn("trade_balance_billions_usd", F.col("trade_balance_usd") / F.lit(1_000_000_000.0))
    .withColumn("intra_project_trade_share_pct", F.when(F.col("total_trade_usd") > 0, F.col("intra_project_trade_usd") / F.col("total_trade_usd") * 100.0))
)

top_partner_window = Window.partitionBy("reporter_iso3", "year").orderBy(F.desc("total_trade_usd"), F.asc("counterpart_iso3"))
top_partner_df = (
    trade_fact
    .where(~F.col("is_world_aggregate"))
    .withColumn("partner_rank", F.row_number().over(top_partner_window))
    .where(F.col("partner_rank") == 1)
    .select(
        F.col("reporter_iso3").alias("top_partner_country_iso3"),
        F.col("year").alias("top_partner_year"),
        F.col("counterpart_iso3").alias("top_partner_iso3"),
        F.col("counterpart_name").alias("top_partner_name"),
        F.col("counterpart_group_code").alias("top_partner_group_code"),
        F.col("total_trade_usd").alias("top_partner_total_trade_usd"),
        F.col("total_trade_billions_usd").alias("top_partner_total_trade_billions_usd"),
        F.col("total_trade_partner_share_pct").alias("top_partner_share_pct"),
    )
)

trade_hhi_df = spark.table("silver.trade_partner_concentration_annual").select(
    F.col("reporter_iso3").alias("hhi_country_iso3"),
    F.col("year").alias("hhi_year"),
    "export_partner_hhi",
    "import_partner_hhi",
    "total_trade_partner_hhi",
)

print(f"Trade annual rows: {trade_year_df.count():,}")
print(f"Top partner rows: {top_partner_df.count():,}")

In [ ]:
# Cell 4 - Build gold.country_year_observatory
coverage = spark.table("audit.country_year_source_coverage")
macro = spark.table("silver.fact_macro_annual").select(
    "country_iso3", "year",
    "gdp_current_usd", "gdp_current_usd_billions", "gdp_per_capita_current_usd",
    "population", "population_millions", "inflation_cpi_pct", "real_gdp_growth_pct_imf",
    "gross_debt_pct_gdp_imf", "gross_debt_usd",
    "government_revenue_pct_gdp_imf", "government_expenditure_pct_gdp_imf",
    "net_lending_borrowing_pct_gdp_imf", "current_account_balance_pct_gdp_imf",
)
acled = spark.table("silver.fact_acled_country_year").select(
    "country_iso3", "year",
    "total_events", "violent_events", "battle_events", "violence_against_civilians_events",
    "explosions_remote_violence_events", "protest_events", "riot_events", "fatalities",
    "events_per_million", "violent_events_per_million", "fatalities_per_million",
)
fsi = spark.table("silver.fact_fsi_annual").select(
    "country_iso3", "year",
    "fsi_rank", "fsi_total_score", "fragility_band",
    "cohesion_score", "economic_score", "political_score", "social_cross_cutting_score",
    "security_apparatus_score", "state_legitimacy_score", "external_intervention_score",
)

country_year_df = (
    coverage.alias("c")
    .join(macro.alias("m"), ["country_iso3", "year"], "left")
    .join(trade_year_df.alias("t"), ["country_iso3", "year"], "left")
    .join(
        trade_hhi_df.alias("h"),
        (F.col("c.country_iso3") == F.col("h.hhi_country_iso3")) & (F.col("c.year") == F.col("h.hhi_year")),
        "left",
    )
    .join(
        top_partner_df.alias("tp"),
        (F.col("c.country_iso3") == F.col("tp.top_partner_country_iso3")) & (F.col("c.year") == F.col("tp.top_partner_year")),
        "left",
    )
    .join(acled.alias("a"), ["country_iso3", "year"], "left")
    .join(fsi.alias("f"), ["country_iso3", "year"], "left")
    .select(
        F.col("c.country_key"),
        F.col("country_iso3"),
        F.col("c.country_name"),
        F.col("c.project_region"),
        F.col("c.analytical_bloc_code"),
        F.col("c.analytical_bloc_name"),
        F.col("year"),
        F.col("c.year_end_date"),
        F.col("c.coverage_level"),
        F.col("c.dashboard_core_ready"),
        F.col("c.full_fragility_context_ready"),
        F.col("c.source_count"),
        F.col("m.gdp_current_usd"),
        F.col("m.gdp_current_usd_billions"),
        F.col("m.gdp_per_capita_current_usd"),
        F.col("m.population"),
        F.col("m.population_millions"),
        F.col("m.inflation_cpi_pct"),
        F.col("m.real_gdp_growth_pct_imf"),
        F.col("m.gross_debt_pct_gdp_imf"),
        F.col("m.gross_debt_usd"),
        F.col("m.government_revenue_pct_gdp_imf"),
        F.col("m.government_expenditure_pct_gdp_imf"),
        F.col("m.net_lending_borrowing_pct_gdp_imf"),
        F.col("m.current_account_balance_pct_gdp_imf"),
        F.col("t.trade_partner_count"),
        F.col("t.exports_fob_usd"),
        F.col("t.imports_cif_usd"),
        F.col("t.total_trade_usd"),
        F.col("t.trade_balance_usd"),
        F.col("t.exports_billions_usd"),
        F.col("t.imports_billions_usd"),
        F.col("t.total_trade_billions_usd"),
        F.col("t.trade_balance_billions_usd"),
        F.col("t.intra_project_trade_usd"),
        F.col("t.intra_project_trade_share_pct"),
        F.col("h.export_partner_hhi"),
        F.col("h.import_partner_hhi"),
        F.col("h.total_trade_partner_hhi"),
        F.col("tp.top_partner_iso3"),
        F.col("tp.top_partner_name"),
        F.col("tp.top_partner_group_code"),
        F.col("tp.top_partner_total_trade_usd"),
        F.col("tp.top_partner_total_trade_billions_usd"),
        F.col("tp.top_partner_share_pct"),
        F.col("a.total_events"),
        F.col("a.violent_events"),
        F.col("a.battle_events"),
        F.col("a.violence_against_civilians_events"),
        F.col("a.explosions_remote_violence_events"),
        F.col("a.protest_events"),
        F.col("a.riot_events"),
        F.col("a.fatalities"),
        F.col("a.events_per_million"),
        F.col("a.violent_events_per_million"),
        F.col("a.fatalities_per_million"),
        F.col("f.fsi_rank"),
        F.col("f.fsi_total_score"),
        F.col("f.fragility_band"),
        F.col("f.cohesion_score"),
        F.col("f.economic_score"),
        F.col("f.political_score"),
        F.col("f.social_cross_cutting_score"),
        F.col("f.security_apparatus_score"),
        F.col("f.state_legitimacy_score"),
        F.col("f.external_intervention_score"),
    )
    .withColumn("trade_openness_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("total_trade_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("exports_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("exports_fob_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("imports_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("imports_cif_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("trade_balance_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("trade_balance_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

print(f"Country-year rows: {country_year_df.count():,}")
country_year_df.show(10, truncate=False)

In [ ]:
# Cell 5 - Write gold.country_year_observatory
spark.sql("DROP TABLE IF EXISTS gold.country_year_observatory")

(country_year_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.country_year_observatory"))

print("Write complete: gold.country_year_observatory")

In [ ]:
# Cell 6 - Build latest country snapshot with latest available FSI carried as separate fields
snapshot_year = spark.table("gold.country_year_observatory").agg(F.max("year")).first()[0]
print(f"Snapshot year: {snapshot_year}")

latest_snapshot_base_df = spark.table("gold.country_year_observatory").where(F.col("year") == snapshot_year)

fsi_latest_window = Window.partitionBy("country_iso3").orderBy(F.desc("year"))
latest_fsi_df = (
    spark.table("gold.country_year_observatory")
    .where(F.col("fsi_total_score").isNotNull())
    .withColumn("rn", F.row_number().over(fsi_latest_window))
    .where(F.col("rn") == 1)
    .select(
        F.col("country_iso3").alias("fsi_country_iso3"),
        F.col("year").alias("latest_fsi_year"),
        F.col("fsi_rank").alias("latest_fsi_rank"),
        F.col("fsi_total_score").alias("latest_fsi_total_score"),
        F.col("fragility_band").alias("latest_fragility_band"),
        F.col("cohesion_score").alias("latest_cohesion_score"),
        F.col("economic_score").alias("latest_economic_score"),
        F.col("political_score").alias("latest_political_score"),
        F.col("social_cross_cutting_score").alias("latest_social_cross_cutting_score"),
    )
)

latest_snapshot_df = (
    latest_snapshot_base_df.alias("s")
    .join(latest_fsi_df.alias("f"), F.col("s.country_iso3") == F.col("f.fsi_country_iso3"), "left")
    .drop("fsi_country_iso3")
    .withColumn("fsi_is_current_year", F.col("latest_fsi_year") == F.col("year"))
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.country_latest_snapshot")

(latest_snapshot_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.country_latest_snapshot"))

print("Write complete: gold.country_latest_snapshot")
latest_snapshot_df.select(
    "analytical_bloc_code", "country_iso3", "country_name", "year",
    F.round("gdp_current_usd_billions", 2).alias("gdp_billions_usd"),
    F.round("total_trade_billions_usd", 2).alias("trade_billions_usd"),
    "violent_events", "fatalities", "latest_fsi_year",
    F.round("latest_fsi_total_score", 1).alias("latest_fsi_total_score"),
).orderBy("analytical_bloc_code", "country_iso3").show(25, truncate=False)

In [ ]:
# Cell 7 - Build gold.bloc_year_observatory
bloc_year_df = (
    spark.table("gold.country_year_observatory")
    .groupBy("analytical_bloc_code", "analytical_bloc_name", "year")
    .agg(
        F.countDistinct("country_iso3").alias("country_count"),
        F.sum("gdp_current_usd").alias("gdp_current_usd"),
        F.sum("population").alias("population"),
        F.sum("exports_fob_usd").alias("exports_fob_usd"),
        F.sum("imports_cif_usd").alias("imports_cif_usd"),
        F.sum("total_trade_usd").alias("total_trade_usd"),
        F.sum("trade_balance_usd").alias("trade_balance_usd"),
        F.sum("gross_debt_usd").alias("gross_debt_usd"),
        F.sum("total_events").alias("total_events"),
        F.sum("violent_events").alias("violent_events"),
        F.sum("fatalities").alias("fatalities"),
        F.avg("fsi_total_score").alias("avg_fsi_total_score"),
        (F.sum(F.when(F.col("total_trade_partner_hhi").isNotNull(), F.col("total_trade_partner_hhi") * F.col("total_trade_usd")).otherwise(F.lit(0.0)))
         / F.sum(F.when(F.col("total_trade_partner_hhi").isNotNull(), F.col("total_trade_usd")).otherwise(F.lit(0.0)))).alias("avg_total_trade_partner_hhi"),
        F.sum(F.when(F.col("dashboard_core_ready"), F.lit(1)).otherwise(F.lit(0))).alias("dashboard_core_ready_countries"),
        F.sum(F.when(F.col("full_fragility_context_ready"), F.lit(1)).otherwise(F.lit(0))).alias("full_context_ready_countries"),
    )
    .withColumn("gdp_current_usd_billions", F.col("gdp_current_usd") / F.lit(1_000_000_000.0))
    .withColumn("population_millions", F.col("population") / F.lit(1_000_000.0))
    .withColumn("bloc_gdp_per_capita_current_usd", F.when(F.col("population") > 0, F.col("gdp_current_usd") / F.col("population")))
    .withColumn("exports_billions_usd", F.col("exports_fob_usd") / F.lit(1_000_000_000.0))
    .withColumn("imports_billions_usd", F.col("imports_cif_usd") / F.lit(1_000_000_000.0))
    .withColumn("total_trade_billions_usd", F.col("total_trade_usd") / F.lit(1_000_000_000.0))
    .withColumn("trade_balance_billions_usd", F.col("trade_balance_usd") / F.lit(1_000_000_000.0))
    .withColumn("trade_openness_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("total_trade_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("gross_debt_pct_gdp", F.when(F.col("gdp_current_usd") > 0, F.col("gross_debt_usd") / F.col("gdp_current_usd") * 100.0))
    .withColumn("events_per_million", F.when(F.col("population") > 0, F.col("total_events") / F.col("population") * 1_000_000.0))
    .withColumn("violent_events_per_million", F.when(F.col("population") > 0, F.col("violent_events") / F.col("population") * 1_000_000.0))
    .withColumn("fatalities_per_million", F.when(F.col("population") > 0, F.col("fatalities") / F.col("population") * 1_000_000.0))
    .withColumn("source_refresh_at", F.lit(loaded_at))
)

spark.sql("DROP TABLE IF EXISTS gold.bloc_year_observatory")

(bloc_year_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bloc_year_observatory"))

print("Write complete: gold.bloc_year_observatory")
bloc_year_df.orderBy("year", "analytical_bloc_code").show(20, truncate=False)

In [ ]:
# Cell 8 - Build gold.bloc_latest_snapshot
latest_bloc_df = spark.table("gold.bloc_year_observatory").where(F.col("year") == snapshot_year)

spark.sql("DROP TABLE IF EXISTS gold.bloc_latest_snapshot")

(latest_bloc_df.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.bloc_latest_snapshot"))

print("Write complete: gold.bloc_latest_snapshot")
latest_bloc_df.select(
    "analytical_bloc_code", "year", "country_count",
    F.round("gdp_current_usd_billions", 2).alias("gdp_billions_usd"),
    F.round("population_millions", 1).alias("population_millions"),
    F.round("bloc_gdp_per_capita_current_usd", 0).alias("gdp_per_capita_usd"),
    F.round("total_trade_billions_usd", 2).alias("trade_billions_usd"),
    F.round("trade_openness_pct_gdp", 1).alias("trade_openness_pct_gdp"),
    "violent_events", "fatalities",
).orderBy("analytical_bloc_code").show(truncate=False)

In [ ]:
# Cell 9 - Verification and sanity checks
print("Gold table row counts:")
for table_name in ["country_year_observatory", "country_latest_snapshot", "bloc_year_observatory", "bloc_latest_snapshot"]:
    row_count = spark.table(f"gold.{table_name}").count()
    print(f"  gold.{table_name}: {row_count:,}")

print("Country-year coverage:")
spark.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT country_iso3) AS countries,
        MIN(year) AS earliest_year,
        MAX(year) AS latest_year,
        SUM(CASE WHEN dashboard_core_ready THEN 1 ELSE 0 END) AS dashboard_core_ready_rows,
        SUM(CASE WHEN full_fragility_context_ready THEN 1 ELSE 0 END) AS full_context_ready_rows
    FROM gold.country_year_observatory
""").show()

print("Latest country snapshot, ranked by GDP:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_iso3,
        country_name,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        ROUND(total_trade_billions_usd, 2) AS trade_billions_usd,
        ROUND(trade_openness_pct_gdp, 1) AS trade_openness_pct_gdp,
        top_partner_iso3,
        top_partner_name,
        ROUND(top_partner_share_pct, 1) AS top_partner_share_pct,
        violent_events,
        fatalities,
        latest_fsi_year,
        ROUND(latest_fsi_total_score, 1) AS latest_fsi_total_score
    FROM gold.country_latest_snapshot
    ORDER BY gdp_current_usd DESC
""").show(25, truncate=False)

print("Latest bloc snapshot:")
spark.sql("""
    SELECT
        analytical_bloc_code,
        country_count,
        ROUND(gdp_current_usd_billions, 2) AS gdp_billions_usd,
        ROUND(population_millions, 1) AS population_millions,
        ROUND(total_trade_billions_usd, 2) AS trade_billions_usd,
        ROUND(gross_debt_pct_gdp, 1) AS debt_pct_gdp,
        violent_events,
        fatalities,
        dashboard_core_ready_countries,
        full_context_ready_countries
    FROM gold.bloc_latest_snapshot
    ORDER BY analytical_bloc_code
""").show(truncate=False)